# Direct point-to-point dipole interaction

## Purpose

The P2P operator is the exact near-field calculation and the reference against
which every truncated fast multipole expansion is judged. This notebook starts
with axial and transverse analytic cases, then sums a deterministic source
cloud and visualises the field of a single dipole in the $z=0$ plane.

P2P consumes source positions, dipole moments, and a target position. It
returns the optional scalar potential $\phi$ and the magnetic field $H$; it
does not use an expansion or tree.

## Mathematical definition

For $r=x_i-x_j$ and $G(r)=1/(4\pi|r|)$,

$$
\phi_{ij}=\frac{m_j\cdot r}{4\pi |r|^3},\qquad
H_{ij}=\frac{1}{4\pi}\left[
\frac{3r(m_j\cdot r)}{|r|^5}-\frac{m_j}{|r|^3}
\right].
$$

Self-interactions must be skipped whenever a target is also a source.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import cdfmm

try:
    from example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )
except ModuleNotFoundError:
    # This path is used when the kernel starts in the repository root.
    from examples.notebooks.example_utils import (
        direct_fields,
        draw_box_3d,
        error_metrics,
        finish_3d_axes,
        local_fields,
        multipole_fields,
        new_3d_figure,
        nodes_at_level,
        plot_coefficients_by_degree,
        random_unit_vectors,
        relative_error,
        set_axes_equal,
        vec3_to_array,
    )

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True})

## User parameters

In [ ]:
n_sources = 20
random_seed = 42
plane_extent = 2.0
grid_points_per_axis = 23
field_clip_percentile = 85.0

## Axial and transverse analytic cases

In [ ]:
source_position = np.array([0.0, 0.0, 0.0])
dipole_moment = np.array([1.0, 0.0, 0.0])
axial_target = np.array([1.0, 0.0, 0.0])
transverse_target = np.array([0.0, 1.0, 0.0])

constant = 1.0 / (4.0 * np.pi)
expected_axial = {"phi": constant, "H": np.array([2.0 * constant, 0.0, 0.0])}
expected_transverse = {"phi": 0.0, "H": np.array([-constant, 0.0, 0.0])}

axial = cdfmm.p2p_dipole_pair(
    axial_target,
    source_position,
    dipole_moment,
    output="both",
)
transverse = cdfmm.p2p_dipole_pair(
    transverse_target,
    source_position,
    dipole_moment,
    output="both",
)

for name, computed, expected in [
    ("axial", axial, expected_axial),
    ("transverse", transverse, expected_transverse),
]:
    print(f"{name.capitalize()} case")
    print(f"  phi computed / expected: {computed['phi']:.12e} / {expected['phi']:.12e}")
    print(f"  H computed:  {computed['H']}")
    print(f"  H expected:  {expected['H']}")
    print(f"  absolute H error: {np.linalg.norm(computed['H'] - expected['H']):.3e}")

## Multiple deterministic dipoles

In [ ]:
rng = np.random.default_rng(random_seed)
source_positions = rng.uniform(-0.5, 0.5, size=(n_sources, 3))
dipole_moments = rng.normal(size=(n_sources, 3))
target_position = np.array([1.75, -0.40, 0.65])

direct_sum = cdfmm.p2p_dipole_sum(
    target_position,
    source_positions,
    dipole_moments,
    output="both",
)

print(f"Number of sources: {n_sources}")
print(f"Target position: {target_position}")
print(f"Scalar potential: {direct_sum['phi']:.8e}")
print(f"Magnetic field: {direct_sum['H']}")

## Field-vector visualisation

In [ ]:
coordinates = np.linspace(-plane_extent, plane_extent, grid_points_per_axis)
x_grid, y_grid = np.meshgrid(coordinates, coordinates)
field = np.full(x_grid.shape + (3,), np.nan)

for row in range(grid_points_per_axis):
    for column in range(grid_points_per_axis):
        point = np.array([x_grid[row, column], y_grid[row, column], 0.0])
        # Omit a small disc around the singular source point.
        if np.linalg.norm(point - source_position) < 0.12:
            continue
        field[row, column] = cdfmm.p2p_dipole_pair(
            point,
            source_position,
            dipole_moment,
            output="field",
        )["H"]

in_plane_magnitude = np.linalg.norm(field[..., :2], axis=-1)
clip_value = np.nanpercentile(in_plane_magnitude, field_clip_percentile)
scale = np.minimum(1.0, clip_value / in_plane_magnitude)
plot_field = field[..., :2] * scale[..., np.newaxis]

figure, axes = plt.subplots(figsize=(8, 7))
colour = np.log10(in_plane_magnitude)
quiver = axes.quiver(
    x_grid,
    y_grid,
    plot_field[..., 0],
    plot_field[..., 1],
    colour,
    cmap="viridis",
    pivot="mid",
)
axes.scatter(*source_position[:2], marker="*", s=180, color="tab:red", label="dipole")
axes.arrow(0.0, 0.0, 0.35, 0.0, width=0.025, color="tab:red")
axes.set_aspect("equal")
axes.set_xlabel("x")
axes.set_ylabel("y")
axes.set_title("Direct dipole field in the z=0 plane")
axes.legend()
figure.colorbar(quiver, ax=axes, label=r"$\log_{10}|H_{xy}|$")
figure.tight_layout()

## Interactive experiment

In [ ]:
from ipywidgets import FloatSlider, interact


def inspect_pair(moment_angle_degrees=0.0, target_x=1.0, target_y=0.0):
    angle = np.deg2rad(moment_angle_degrees)
    moment = np.array([np.cos(angle), np.sin(angle), 0.0])
    target = np.array([target_x, target_y, 0.0])
    if np.linalg.norm(target) < 1.0e-8:
        print("Move the target away from the singular source point.")
        return

    result = cdfmm.p2p_dipole_pair(
        target,
        source_position,
        moment,
        output="both",
    )
    print(f"moment = {moment}")
    print(f"target = {target}")
    print(f"phi = {result['phi']:.6e}")
    print(f"H = {result['H']}")


interact(
    inspect_pair,
    moment_angle_degrees=FloatSlider(min=0.0, max=360.0, step=15.0, value=0.0),
    target_x=FloatSlider(min=-2.0, max=2.0, step=0.1, value=1.0),
    target_y=FloatSlider(min=-2.0, max=2.0, step=0.1, value=0.0),
)

## What to observe

The axial field is twice the Green-function prefactor and points along the
dipole, whereas the transverse field points against the dipole with half that
magnitude. The vector plot exposes the familiar directional lobes and the
$|r|^{-3}$ field singularity; clipping affects arrow length only, not the
colour diagnostic.